<a href="https://colab.research.google.com/github/syedbeeban/IE/blob/main/02_Solving_Nonlinear_Volterra_Equations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

I would be happy to show you how to adapt these methods. Moving to a **nonlinear Volterra integral equation** introduces two main changes:

1. **Nonlinearity:** The unknown function $u(x)$ might be squared, exponentiated, or inside another function (e.g., $u(t)^2$).
2. **Variable Upper Limit:** Unlike a Fredholm equation (where integration limits are fixed constants), a Volterra equation has a variable upper limit of integration (usually $x$).

To handle the variable upper limit numerically, we map the integration interval from 0 to $x$ into a standard interval of 0 to 1 using a dummy variable $\tau$.
If $t = x\tau$, then $dt = x d\tau$.

Let's solve the following **nonlinear Volterra equation**:

$$u(x) = x - \frac{1}{4}x^4 + \int_0^x t \cdot u(t)^2 dt$$

The exact analytical solution to this is **$u(x) = x$**.

---

### 1. Gradient-Based Optimization (BFGS)

Here, we adapt the objective function. Because the upper limit of the integral is $x$, the integration grid $t$ changes for every collocation point $x$.

In [ ]:
import numpy as np
from scipy.optimize import minimize

# Collocation points to evaluate the residual
x_points = np.linspace(0.1, 1, 20) # Avoid 0 to prevent trivial 0=0 evaluations

# Standardized integration points from 0 to 1
tau_points = np.linspace(0, 1, 50)
dtau = tau_points[1] - tau_points[0]

def objective_volterra(c):
    """
    Minimizes the residual of the nonlinear Volterra equation.
    c: coefficients [c0, c1, c2] for u(x) = c0 + c1*x + c2*x^2
    """
    total_error = 0

    for x in x_points:
        # Calculate u(x)
        u_x = c[0] + c[1]*x + c[2]*x**2

        # Map tau to actual t values for this specific x
        t_mapped = x * tau_points
        dt = x * dtau

        # Calculate u(t) at mapped points
        u_t = c[0] + c[1]*t_mapped + c[2]*t_mapped**2

        # Evaluate the nonlinear integral: integral of t * u(t)^2 dt
        integral_val = np.sum(t_mapped * (u_t**2) * dt)

        # Left Hand Side and Right Hand Side of the Volterra equation
        lhs = u_x - integral_val
        rhs = x - 0.25 * x**4

        total_error += (lhs - rhs)**2

    return total_error

# Run optimization
initial_guess = [0.1, 0.5, 0.1] # Start slightly away from the answer
result = minimize(objective_volterra, initial_guess, method='BFGS')

print("Nonlinear Volterra - Gradient-Based (BFGS):")
print(f"Optimized Coefficients (c0, c1, c2): {result.x}")
print("Expected: ~[0, 1, 0] because u(x) = x")

---

### 2. Global Optimization (Differential Evolution)

The beauty of global optimization is that the solver logic remains completely identical to how we set up the BFGS approach. We just pass the non-linear objective function into the evolutionary algorithm, which prevents the squared term $u(t)^2$ from trapping the solver in a local minimum.

In [ ]:
import numpy as np
from scipy.optimize import differential_evolution

# (Reusing x_points, tau_points, dtau, and objective_volterra from above)

# Define search boundaries for the coefficients
bounds = [(-2, 2), (-2, 2), (-2, 2)]

# Run Differential Evolution
result_de = differential_evolution(objective_volterra, bounds, strategy='best1bin', seed=42)

print("Nonlinear Volterra - Differential Evolution:")
print(f"Optimized Coefficients (c0, c1, c2): {result_de.x}")
print("Expected: ~[0, 1, 0]")

---

### 3. Neural Network Optimization (PINNs)

For a neural network in PyTorch, handling the variable upper limit $x$ requires us to use tensor broadcasting to calculate the integral for all $x$ points simultaneously. This makes the code highly efficient.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class VolterraNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 20),
            nn.Tanh(),
            nn.Linear(20, 1)
        )

    def forward(self, x):
        return self.net(x)

model = VolterraNet()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Set up tensors
x_tensor = torch.linspace(0.1, 1.0, 30).view(-1, 1) # Shape: (30, 1)
tau_tensor = torch.linspace(0.0, 1.0, 50).view(-1, 1) # Shape: (50, 1)
dtau = tau_tensor[1] - tau_tensor[0]

epochs = 1500
for epoch in range(epochs):
    optimizer.zero_grad()

    # Evaluate network at x
    u_x = model(x_tensor)

    # Create the mapped 't' matrix for all combinations of x and tau
    # t_matrix shape: (30, 50) - each row corresponds to t values for a specific x
    t_matrix = x_tensor @ tau_tensor.T

    # Flatten t_matrix to pass through the neural network, then reshape back
    u_t_flat = model(t_matrix.view(-1, 1))
    u_t_matrix = u_t_flat.view(x_tensor.shape[0], tau_tensor.shape[0])

    # Calculate the nonlinear integral: sum of (t * u(t)^2 * dt)
    # Note: dt is x * dtau for each specific x
    dt_matrix = x_tensor * dtau
    integral_val = torch.sum(t_matrix * (u_t_matrix**2) * dt_matrix, dim=1).view(-1, 1)

    # Calculate Residual Loss
    lhs = u_x - integral_val
    rhs = x_tensor - 0.25 * (x_tensor**4)

    loss = torch.mean((lhs - rhs)**2)

    loss.backward()
    optimizer.step()

    if epoch % 500 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.6f}")

print("\nNonlinear Volterra - Neural Network (Adam):")
test_x = torch.tensor([[0.5], [0.8]])
predicted_u = model(test_x).detach().numpy()
print(f"Prediction at x=0.5: {predicted_u[0][0]:.4f} (Expected: 0.5)")
print(f"Prediction at x=0.8: {predicted_u[1][0]:.4f} (Expected: 0.8)")

---

Would you like to explore how to add regularization to these methods to handle noisy data (like solving an inverse problem where the right-hand side is measured from sensors)?